In [ ]:
import numpy as np
import pandas as pd

import cv2
from PIL import Image
from torchvision import transforms

import tempfile
import os
import joblib
from ultralytics import YOLO

from segmentation.modelsUnet import UNet4Levels, preprocess_image, predict_masks
from segmentation.features import extract_features
from classification.models import CytologyClassifier, predict_label

from app.backend.helpers import *  

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from dotenv import load_dotenv
from pathlib import Path


load_dotenv()

CLASS_NAMES = os.getenv("CLASS_NAMES", "HSIL,LSIL,NSIL").split(",")
ARCHITECTURE = os.getenv("ARCHITECTURE", "resnet18")
CNN_MODEL_PATH = os.getenv("CNN_MODEL_PATH", r"C:\Users\aleks\OneDrive\Documents\inzynierka\classification\classification_models\resnet18\16_0_0001_50_1110.pth")
UNET_MODEL_PATH = os.getenv("UNET_MODEL_PATH", r"C:\Users\aleks\OneDrive\Documents\inzynierka\segmentation\unet4_cell_nucleus_4_50_1310.pth")
THRESHOLD_NUCLEI = float(os.getenv("THRESHOLD_NUCLEI", 0.7))
THRESHOLD_CELLS = float(os.getenv("THRESHOLD_CELLS", 0.3))
YOLO_MODEL_PATH = os.getenv("YOLO_MODEL_PATH", r"C:\Users\aleks\OneDrive\Documents\inzynierka\yolo_models\models\yolo_detector_2107_100_20_16_7682\weights\best.pt")
ML_MODEL_PATH = os.getenv("ML_MODEL_PATH", r'C:\Users\aleks\OneDrive\Documents\inzynierka\segmentation\models_paths\best_model_RandomForest_new_unet.pkl')
FUSED_MODEL_PATH = os.getenv("FUSED_MODEL_PATH", r"C:\Users\aleks\OneDrive\Documents\inzynierka\segmentation\models_paths\joined\model_probs_1410_SVM2.pkl")
API_KEY = os.getenv("API_KEY", os.getenv("api_key", ''))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


unet = UNet4Levels(in_channels=3, out_channels=2)
unet.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=device))
unet.to(device).eval()

yolo = YOLO(YOLO_MODEL_PATH)

ml_model = joblib.load(ML_MODEL_PATH)
label_encoder = ml_model['label_encoder']

fused_model = joblib.load(FUSED_MODEL_PATH)
label_encode_fused = fused_model['label_encoder']

cnn_classifier = CytologyClassifier(num_classes=len(CLASS_NAMES), architecture=ARCHITECTURE)
cnn_classifier.load(CNN_MODEL_PATH)
cnn_classifier.model.eval().to(device)


def get_info(image_path, show_image=True):
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    results = yolo(image_path, conf=0.25, iou=0.5, agnostic_nms=False)
    boxes_tensor = results[0].boxes.xyxy.cpu()
    classes_tensor = results[0].boxes.cls.cpu().int()

    boxes_array = boxes_tensor.numpy()
    keep_ids = nms_keep_largest_box(boxes_array, iou_thresh=0.4)

    boxes_tensor = boxes_tensor[keep_ids]
    classes_tensor = classes_tensor[keep_ids]

    cell_mask = (classes_tensor == 0) | (classes_tensor == 1) | (classes_tensor == 2)
    boxes = boxes_tensor[cell_mask].numpy().astype(int)

    predicted_classes_cnn = {}
    predicted_classed_ml = {}
    predicted_classed_fused = {}
    features_list = {}
    probs = {}
    crop_paths = {}
    probs_list = []

    bbox_image_path = None
    if show_image:
        fig, ax = plt.subplots(figsize=(7.68, 5.12), dpi=100)
        ax.imshow(image_np)
        for idx, (x1, y1, x2, y2) in enumerate(boxes):
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2,
                edgecolor= '#f7c873',
                facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(
                x1, max(0, y1 - 5), str(idx),
                color='#3a5ba0',  
                fontsize=12, weight='bold',
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='none')
            )
        ax.set_axis_off()
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp_bbox:
            fig.savefig(tmp_bbox.name, bbox_inches='tight', pad_inches=0, dpi=100)
            bbox_image_path = tmp_bbox.name
        plt.close(fig)
        im = Image.open(bbox_image_path)
        im = im.resize((768, 512), Image.LANCZOS)
        im.save(bbox_image_path)

    for idx, (x1, y1, x2, y2) in enumerate(boxes):
        crop = image_np[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        yolo_class = None
        for i, box in enumerate(boxes_tensor.numpy().astype(int)):
            if np.array_equal(box, [x1, y1, x2, y2]):
                yolo_class = int(classes_tensor[i].item())
                break

        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp_crop:
            cv2.imwrite(tmp_crop.name, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
            tmp_path = tmp_crop.name
            crop_paths[idx] = tmp_path

        if yolo_class == 1:
            predicted_classes_cnn[idx] = 'HSIL/LSIL_group'
            predicted_classed_fused[idx] = 'HSIL/LSIL_group'
            probs_list.append([0.7, 0.3, 0.0])
            continue

        _, tensor = preprocess_image(tmp_path)
        masks = predict_masks(unet, tensor, device, threshold_nuclei=THRESHOLD_NUCLEI, threshold_cell=THRESHOLD_CELLS)
        mask_nucleus = cv2.resize(masks[1], (crop.shape[1], crop.shape[0]))
        best_nucleus = select_best_nucleus(mask_nucleus, crop.shape[:2])
        mask_cell = cv2.resize(masks[0], (crop.shape[1], crop.shape[0])) * 255

        features = extract_features(best_nucleus, mask_cell)
        
        if features is None or len(features) == 0:
            print(f"Pominięto komórkę {idx} - brak features")
            continue
            
        if all(value == 0 for value in features.values()):
            print(f"Pominięto komórkę {idx} - wszystkie features równe 0")
            continue
            
        features_list[idx] = features

        predict_class_ml = predict_ml(ml_model['model'], label_encoder, features)
        predicted_classed_ml[idx] = predict_class_ml

        predict_class_cnn = predict_label(cnn_classifier, crop)
        predicted_classes_cnn[idx] = predict_class_cnn[0]

        rf_predictions = predict_fused_func(fused_model['model'], label_encode_fused, cnn_classifier,   ml_model['model'], unet, device, tmp_path)
        predicted_classed_fused[idx] = rf_predictions

        probs_fused = predict_fused_func(
            fused_model['model'], label_encode_fused, cnn_classifier, ml_model['model'], unet, device, tmp_path, probs_output=True
        )
        probs[idx] = {
            'fused': probs_fused
        }
        probs_list.append(probs_fused)
    
    all_indices = set(predicted_classes_cnn.keys()) | set(predicted_classed_ml.keys()) | set(predicted_classed_fused.keys())
    rows = []
    for idx in sorted(all_indices):
        rows.append({
            "idx": idx,
            "predict_vgg": predicted_classes_cnn.get(idx, None),
            "predict_gbm": predicted_classed_ml.get(idx, None),
            "predicted_classed_fused": predicted_classed_fused.get(idx, None),
        })

    df_preds = pd.DataFrame(rows)

    return features_list, predicted_classed_fused, probs_list, df_preds, bbox_image_path, crop_paths

    


c:\Users\aleks\OneDrive\Documents\inzynierka\inz\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\aleks\OneDrive\Documents\inzynierka\inz\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [52]:
import pandas as pd 

def get_metric(folder_path, class_name):
    class_names = CLASS_NAMES
    success_count_mean = 0
    success_count_max = 0
    success_count_weight = 0
    total_count = 0
    filenames = os.listdir(folder_path)
    true_classes = [class_name] * len(os.listdir(folder_path))
    predicted_classes_mean = []
    predicted_classes_max = []
    predicted_classes_weight = []
    mean_vectors = []
    max_wectors = []
    weight_wectors = []
    probs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            image_path = os.path.join(folder_path, filename)
            features_list, predicted_classed_fused, probs_list, df_preds, bbox_image_path, crop_pat = get_info(image_path, show_image=False)
            probs.append(probs_list)
            if len(probs_list) == 0:
                print(f"⚠️ Warning: No cells detected in {filename} - skipping")
                mean_vectors.append(None)
                predicted_classes_mean.append(None)
                max_wectors.append(None)
                predicted_classes_max.append(None)
                weight_wectors.append(None)
                predicted_classes_weight.append(None)                
                continue

            mean_vector = np.mean(probs_list, axis=0)
            predicted_class_mean_vector = class_names[np.argmax(mean_vector)]
            mean_vectors.append(mean_vector)
            predicted_classes_mean.append(predicted_class_mean_vector)

            slide_max = np.max(probs_list, axis=0)
            predicted_class_slide_max = class_names[np.argmax(slide_max)]
            max_wectors.append(slide_max)
            predicted_classes_max.append(predicted_class_slide_max)

            weights = np.max(probs_list, axis=1)
            weighted_wector = np.average(probs_list, axis=0, weights=weights)
            predicted_class_slide_weighted = class_names[np.argmax(weighted_wector)]
            weight_wectors.append(weighted_wector)
            predicted_classes_weight.append(predicted_class_slide_weighted)


            if predicted_class_mean_vector == class_name:
                success_count_mean += 1
            if predicted_class_slide_max == class_name:
                success_count_max += 1
            if predicted_class_slide_weighted == class_name:
                success_count_weight += 1
            total_count += 1
            

    accuracy_mean = success_count_mean / total_count if total_count > 0 else 0
    accuracy_max = success_count_max / total_count if total_count > 0 else 0
    accuracy_weight = success_count_weight / total_count if total_count > 0 else 0

    df_accuracy = pd.DataFrame({
        'Method': ['Mean Vector', 'Slide Max', 'Weighted Average'],
        'Accuracy': [accuracy_mean, accuracy_max, accuracy_weight]
    })
    df = pd.DataFrame({'Filename': filenames, 
                       'True Class': true_classes, 
                       'Predicted Class_Mean': predicted_classes_mean, 
                       'Mean Vector': mean_vectors, 
                       'Predicted Class_Max': predicted_classes_max, 
                       'Max Vector': max_wectors, 
                       'Predicted Class_Weighted': predicted_classes_weight, 
                       'Weight Vector': weight_wectors, 
                       'Probs': probs})

    return df_accuracy, df

In [53]:
accuracy_h, df_h = get_metric(r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 40", "HSIL")

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 40\10b.bmp: 480x768 14 cells, 2 HSIL_groups, 70.5ms
Speed: 4.1ms preprocess, 70.5ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 14 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 40\11b.bmp: 480x768 7 cells, 1 HSIL_group, 82.3ms
Speed: 3.3ms preprocess, 82.3ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 40\12b.bmp: 480x768 19 cells, 75.1ms
Speed: 3.4ms preprocess, 75.1ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 40\13b.bmp: 480x768 7 cells, 2 HSIL_groups, 69.8ms
Speed: 3.4ms preprocess, 69.8ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzy

In [54]:
accuracy_h

,Method,Accuracy
0,Mean Vector,0.677419
1,Slide Max,0.645161
2,Weighted Average,0.677419


In [59]:
df_h.to_excel('HSIL_pow40.xlsx', index=False) 

In [60]:
accuracy_l, df_l = get_metric(r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow 40", "LSIL")


image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow 40\10b.bmp: 480x768 2 cells, 113.8ms
Speed: 6.2ms preprocess, 113.8ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 1 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow 40\11b.bmp: 480x768 12 cells, 57.1ms
Speed: 3.6ms preprocess, 57.1ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow 40\12b.bmp: 480x768 6 cells, 70.3ms
Speed: 3.5ms preprocess, 70.3ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow 40\13b.bmp: 480x768 2 cells, 79.9ms
Speed: 3.1ms preprocess, 79.9ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow 40\14b.bmp:

In [61]:
accuracy_l

,Method,Accuracy
0,Mean Vector,0.823529
1,Slide Max,0.705882
2,Weighted Average,0.823529


In [62]:
df_l.to_excel('LSIL_pow40.xlsx', index=False)

In [63]:
accuracy_n, df_n = get_metric(r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow 40", "NSIL")

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow 40\10b.bmp: 480x768 3 cells, 70.1ms
Speed: 3.9ms preprocess, 70.1ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow 40\11b.bmp: 480x768 1 cell, 68.5ms
Speed: 3.4ms preprocess, 68.5ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow 40\12b.bmp: 480x768 1 cell, 65.6ms
Speed: 3.9ms preprocess, 65.6ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow 40\13b.bmp: 480x768 1 cell, 91.5ms
Speed: 3.6ms preprocess, 91.5ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow 40\14b.bmp: 480x768 2 cells, 85.9ms
Speed: 3.4ms preprocess, 85.9ms

In [64]:
accuracy_n

,Method,Accuracy
0,Mean Vector,0.979866
1,Slide Max,0.966443
2,Weighted Average,0.979866


In [66]:
df_n.to_excel('NSIL_pow40.xlsx', index=False)

## pow 10

In [67]:
accuracy_h_10, df_h_10 = get_metric(r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 10", "HSIL")


image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 10\10a.bmp: 480x768 84 cells, 1 HSIL_group, 84.5ms
Speed: 2.9ms preprocess, 84.5ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 12 - wszystkie features równe 0
Pominięto komórkę 23 - wszystkie features równe 0
Pominięto komórkę 24 - wszystkie features równe 0
Pominięto komórkę 40 - wszystkie features równe 0
Pominięto komórkę 49 - wszystkie features równe 0
Pominięto komórkę 56 - wszystkie features równe 0
Pominięto komórkę 73 - wszystkie features równe 0
Pominięto komórkę 80 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 10\11a.bmp: 480x768 1 cell, 1 HSIL_group, 71.5ms
Speed: 4.0ms preprocess, 71.5ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\HSIL\pow 10\12a.bmp: 480x768 75 cells, 1 HSIL_group, 84.7ms
Speed: 3.4ms

In [69]:
accuracy_h_10

,Method,Accuracy
0,Mean Vector,0.322581
1,Slide Max,0.258065
2,Weighted Average,0.322581


In [68]:
df_h_10

,Filename,True Class,Predicted Class_Mean,Mean Vector,Predicted Class_Max,Max Vector,Predicted Class_Weighted,Weight Vector,Probs
0,10a.bmp,HSIL,HSIL,"[0.5227542476937705, 0.1174510408801949, 0.359...",NSIL,"[0.9960459870009604, 0.9778753189343549, 0.996...",HSIL,"[0.5253506170486192, 0.10207956648632621, 0.37...","[[0.0020183341409103026, 0.0027077560600981885..."
1,11a.bmp,HSIL,HSIL,"[0.9973314773855628, 0.0010005094993395967, 0....",HSIL,"[0.9973314773855628, 0.0010005094993395967, 0....",HSIL,"[0.9973314773855628, 0.0010005094993395967, 0....","[[0.9973314773855628, 0.0010005094993395967, 0..."
2,12a.bmp,HSIL,NSIL,"[0.3121626876131099, 0.16006808486262494, 0.52...",NSIL,"[0.9968947384554893, 0.9593551695462026, 0.996...",NSIL,"[0.29400672055990973, 0.1413113080639338, 0.56...","[[0.0020317593225116573, 0.0027370758193978792..."
3,13a.bmp,HSIL,LSIL,"[0.17363059822626253, 0.7881723437199007, 0.03...",LSIL,"[0.8442267014510018, 0.9959362996601453, 0.214...",LSIL,"[0.14166682614185197, 0.8250205207680229, 0.03...","[[0.022327821993147312, 0.7636228234763742, 0...."
4,14a.bmp,HSIL,NSIL,"[0.3416227433667219, 0.055184947917273186, 0.6...",NSIL,"[0.9728497217879941, 0.12571734147490218, 0.99...",NSIL,"[0.3323152495780874, 0.04779886793446786, 0.61...","[[0.5728758573786022, 0.12571734147490218, 0.3..."
...,...,...,...,...,...,...,...,...,...
57,63a.bmp,HSIL,LSIL,"[0.3439259938129396, 0.35579326182462623, 0.30...",LSIL,"[0.9972525889146941, 0.9979969953792543, 0.978...",HSIL,"[0.35401832412565715, 0.34923855403773824, 0.2...","[[0.0325791284224904, 0.40983630889710376, 0.5..."
58,6a.bmp,HSIL,HSIL,"[0.5747537525146331, 0.19225271713609673, 0.23...",HSIL,"[0.9742814562224982, 0.9062253033106684, 0.807...",HSIL,"[0.6102719820449426, 0.18868130023647878, 0.20...","[[0.1310679434197145, 0.21703796197955366, 0.6..."
59,7a.bmp,HSIL,NSIL,"[0.32331789615041906, 0.06947656015614005, 0.6...",NSIL,"[0.9292001960830063, 0.26389209210942893, 0.99...",NSIL,"[0.29303027733975434, 0.05584225336336577, 0.6...","[[0.002011131207089998, 0.002612525007487855, ..."
60,8a.bmp,HSIL,HSIL,"[0.7487135744176706, 0.06150360994356207, 0.18...",HSIL,"[0.9390603424719104, 0.14363230954180486, 0.59...",HSIL,"[0.7961614519139161, 0.05354694190586874, 0.15...","[[0.8603073670541972, 0.041532249570650236, 0...."


In [70]:
accuracy_l_10, df_l_10 = get_metric(r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow. 10", "LSIL")
print(accuracy_l_10)


image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow. 10\10a.bmp: 480x768 3 cells, 113.1ms
Speed: 3.1ms preprocess, 113.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 2 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow. 10\11a.bmp: 480x768 80 cells, 4 HSIL_groups, 93.8ms
Speed: 3.3ms preprocess, 93.8ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 63 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\LSIL\pow. 10\13a.bmp: 480x768 47 cells, 79.2ms
Speed: 4.6ms preprocess, 79.2ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 5 - wszystkie features równe 0
Pominięto komórkę 8 - wszystkie features równe 0
Pominięto komórkę 10 - wszystkie features równe 0
Pominięto komórkę 12 - wszystkie features równe 0
Pominięto komórkę 29 - wszystkie featur

In [71]:
accuracy_n_10, df_n_10 = get_metric(r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow. 10", "NSIL")
print(accuracy_n_10)


image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow. 10\10a.bmp: 480x768 35 cells, 2 HSIL_groups, 87.1ms
Speed: 3.6ms preprocess, 87.1ms inference, 3.3ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 29 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow. 10\11a.bmp: 480x768 2 cells, 67.3ms
Speed: 3.9ms preprocess, 67.3ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow. 10\12a.bmp: 480x768 2 cells, 66.6ms
Speed: 3.3ms preprocess, 66.6ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\NSIL\pow. 10\13a.bmp: 480x768 12 cells, 67.2ms
Speed: 3.7ms preprocess, 67.2ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 768)

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_slides\N

## WAGI DLA HSIL 

In [7]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score, 
                             precision_score, classification_report, confusion_matrix)
from itertools import product
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [8]:
def class_weighted_pooling(cell_probs, class_weights=[3.0, 1.5, 1.0], pooling='max'):
    if len(cell_probs) == 0:
        return None
    
    weighted_probs = cell_probs * np.array(class_weights)
    
    if pooling == 'max':
        slide_probs_weighted = np.max(weighted_probs, axis=0)
    elif pooling == 'mean':
        slide_probs_weighted = np.mean(weighted_probs, axis=0)
    else:
        raise ValueError(f"Unknown pooling: {pooling}")
    
    slide_probs = slide_probs_weighted / slide_probs_weighted.sum()
    return slide_probs

In [9]:
def load_cell_probabilities_from_folder(folder_path, class_name, class_to_idx):
    slides = []
    labels = []
    filenames = []
    
    if not os.path.exists(folder_path):
        print(f"⚠️  Folder nie istnieje: {folder_path}")
        return slides, labels, filenames
    
    files = [f for f in os.listdir(folder_path) if f.endswith(('.bmp'))]
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        features_list, predicted_classed_fused, cell_probs, df_preds, bbox_image_path, crop_pat = get_info(file_path, show_image=False)

        if len(cell_probs) == 0:
                print(f"⚠️  Brak komórek w {filename}, pomijam...")
                continue
        slides.append(cell_probs)
        labels.append(class_to_idx[class_name])
        filenames.append(filename)
            
    return slides, labels, filenames

In [10]:
def load_all_data(base_path, class_names):

    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    
    train_slides, train_labels, train_files = [], [], []
    test_slides, test_labels, test_files = [], [], []
    
    # Wczytaj train
    print("\n📂 Wczytuję dane treningowe...")
    train_path = os.path.join(base_path, 'train')
    for class_name in class_names:
        folder_path = os.path.join(train_path, class_name)
        slides, labels, files = load_cell_probabilities_from_folder(
            folder_path, class_name, class_to_idx
        )
        train_slides.extend(slides)
        train_labels.extend(labels)
        train_files.extend(files)
        print(f"  {class_name}: {len(slides)} slajdów")
    
    # Wczytaj test
    print("\n📂 Wczytuję dane testowe...")
    test_path = os.path.join(base_path, 'test')
    for class_name in class_names:
        folder_path = os.path.join(test_path, class_name)
        slides, labels, files = load_cell_probabilities_from_folder(
            folder_path, class_name, class_to_idx
        )
        test_slides.extend(slides)
        test_labels.extend(labels)
        test_files.extend(files)
        print(f"  {class_name}: {len(slides)} slajdów")
    
    train_data = {
        'slides': train_slides,
        'labels': np.array(train_labels),
        'filenames': train_files
    }
    
    test_data = {
        'slides': test_slides,
        'labels': np.array(test_labels),
        'filenames': test_files
    }
    
    print(f"\n✓ Wczytano:")
    print(f"  Train: {len(train_slides)} slajdów")
    print(f"  Test: {len(test_slides)} slajdów")
    
    return train_data, test_data

In [11]:
def calculate_metrics(y_true, y_pred, class_names):
    """Oblicz wszystkie metryki"""
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Recall per class
    recall_per_class = recall_score(y_true, y_pred, labels=[0, 1, 2], 
                                    average=None, zero_division=0)
    
    hsil_recall = recall_per_class[0] if len(recall_per_class) > 2 else 0.0
    
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'hsil_recall': hsil_recall,
        'recall_per_class': recall_per_class
    }

In [12]:
def cross_validated_weight_search(train_data, class_names, n_splits=5):
    """
    K-fold Cross-Validation Grid Search dla wag klas i metod poolingu
    
    Testuje zarówno MAX jak i MEAN pooling
    
    Args:
        train_data: dict z 'slides' i 'labels'
        class_names: lista nazw klas ['NILM', 'LSIL', 'HSIL']
        n_splits: liczba foldów w CV
    
    Returns:
        best_config: dict z najlepszą konfiguracją (weights + pooling)
        results_df: DataFrame z wszystkimi wynikami
    """
    print("\n" + "="*70)
    print("CROSS-VALIDATION GRID SEARCH - MAX vs MEAN POOLING")
    print("="*70)
    
    # Przestrzeń przeszukiwania wag
    # UWAGA: Wagi w kolejności [w_HSIL, w_LSIL, w_NILM]
    hsil_weights = [2.0, 3.0, 4.0, 5.0, 7.0]
    lsil_weights = [1.0, 1.5, 2.0, 2.5, 3.0]
    nilm_weights = [1.0]  # NILM bazowy
    
    pooling_methods = ['max', 'mean']
    
    all_combinations = list(product(hsil_weights, lsil_weights, nilm_weights, pooling_methods))
    print(f"\nLiczba kombinacji do przetestowania: {len(all_combinations)}")
    print(f"  - Metody poolingu: {pooling_methods}")
    print(f"  - Wagi HSIL: {hsil_weights}")
    print(f"  - Wagi LSIL: {lsil_weights}")
    print(f"  - Wagi NILM: {nilm_weights}")
    print(f"\nFolds: {n_splits}")
    print(f"Łącznie ewaluacji: {len(all_combinations) * n_splits}")
    
    # K-fold split
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    results = []
    
    # Grid search
    for w_hsil, w_lsil, w_nilm, pooling in tqdm(all_combinations, desc="Grid Search"):
        class_weights = [w_hsil, w_lsil, w_nilm]  # [HSIL, LSIL, NILM]
        
        fold_metrics = {
            'accuracy': [],
            'f1_macro': [],
            'hsil_recall': []
        }
        
        # Cross-validation
        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_data['slides'], 
                                                                   train_data['labels'])):
            # Split data
            val_slides = [train_data['slides'][i] for i in val_idx]
            val_labels = train_data['labels'][val_idx]
            
            # Predykcje na walidacji
            val_preds = []
            for cell_probs in val_slides:
                slide_probs = class_weighted_pooling(cell_probs, class_weights, pooling)
                if slide_probs is not None:
                    val_preds.append(np.argmax(slide_probs))
                else:
                    val_preds.append(2)  
            
            val_preds = np.array(val_preds)
            
            # Metryki
            metrics = calculate_metrics(val_labels, val_preds, class_names)
            
            fold_metrics['accuracy'].append(metrics['accuracy'])
            fold_metrics['f1_macro'].append(metrics['f1_macro'])
            fold_metrics['hsil_recall'].append(metrics['hsil_recall'])
        
        # Średnia i odchylenie standardowe z foldów
        results.append({
            'pooling': pooling,
            'w_hsil': w_hsil,
            'w_lsil': w_lsil,
            'w_nilm': w_nilm,
            'weights': class_weights,
            'mean_accuracy': np.mean(fold_metrics['accuracy']),
            'std_accuracy': np.std(fold_metrics['accuracy']),
            'mean_f1': np.mean(fold_metrics['f1_macro']),
            'std_f1': np.std(fold_metrics['f1_macro']),
            'mean_hsil_recall': np.mean(fold_metrics['hsil_recall']),
            'std_hsil_recall': np.std(fold_metrics['hsil_recall']),

        })
    
    # Stwórz DataFrame z wynikami
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('mean_f1', ascending=False)
    
    # Najlepsza konfiguracja
    best = results_df.iloc[0]
    best_config = {
        'pooling': best['pooling'],
        'weights': [best['w_hsil'], best['w_lsil'], best['w_nilm']],
        'metrics': {
            'accuracy': best['mean_accuracy'],
            'f1_macro': best['mean_f1'],
            'hsil_recall': best['mean_hsil_recall'],
        }
    }
    
    # Wyświetl wyniki
    print("\n" + "="*70)
    print("WYNIKI")
    print("="*70)
    print(f"\n🏆 NAJLEPSZA KONFIGURACJA:")
    print(f"  Pooling: {best['pooling'].upper()}")
    print(f"  Wagi [HSIL, LSIL, NILM]: {best_config['weights']}")
    print(f"\nMetryki (średnia ± std z {n_splits} foldów):")
    print(f"  Accuracy:     {best['mean_accuracy']:.4f} ± {best['std_accuracy']:.4f}")
    print(f"  F1 Macro:     {best['mean_f1']:.4f} ± {best['std_f1']:.4f}")
    print(f"  HSIL Recall:  {best['mean_hsil_recall']:.4f} ± {best['std_hsil_recall']:.4f}")
    
    print(f"\n📊 Top 10 konfiguracji:")
    print(results_df.head(10)[['pooling', 'weights', 'mean_accuracy', 'mean_f1', 
                                'mean_hsil_recall']])
    
    # Porównanie MAX vs MEAN
    print(f"\n📊 Porównanie MAX vs MEAN (najlepsze wyniki):")
    for pooling_method in ['max', 'mean']:
        best_for_method = results_df[results_df['pooling'] == pooling_method].iloc[0]
        print(f"\n  {pooling_method.upper()}:")
        print(f"    Wagi: {[best_for_method['w_hsil'], best_for_method['w_lsil'], best_for_method['w_nilm']]}")
        print(f"    HSIL Recall:  {best_for_method['mean_hsil_recall']:.4f}")
        print(f"    Accuracy:     {best_for_method['mean_accuracy']:.4f}")
    
    return best_config, results_df

In [ ]:
BASE_PATH = r"C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_dataset"
CLASS_NAMES = ['HSIL', 'LSIL', 'NSIL']  # Dla etykiet i nazw folderów
N_SPLITS = 5  # liczba foldów w CV
    
print("="*70)
print("OPTYMALIZACJA WAG + PORÓWNANIE MAX vs MEAN POOLING")
print("="*70)
print(f"\nŚcieżka bazowa: {BASE_PATH}")
print(f"Klasy: {CLASS_NAMES}")
print(f"\n⚠️  UWAGA: Format prawdopodobieństw w plikach: [p_HSIL, p_LSIL, p_NSIL]")
    
# 1. Wczytaj dane
train_data, test_data = load_all_data(BASE_PATH, CLASS_NAMES)
    
if len(train_data['slides']) == 0:
    print("❌ Brak danych treningowych! Sprawdź ścieżki i format plików.")
    
    
# 2. Cross-validation grid search na train (MAX vs MEAN)
best_config, results_df = cross_validated_weight_search(
    train_data, CLASS_NAMES, n_splits=N_SPLITS
    )
    
# 3. Zapisz wyniki CV
results_df.to_csv('cv_grid_search_max_vs_mean.csv', index=False)
print(f"\n✓ Zapisano wyniki CV do: cv_grid_search_max_vs_mean_HSIL_LSIL_GROUP.csv")

OPTYMALIZACJA WAG + PORÓWNANIE MAX vs MEAN POOLING

Ścieżka bazowa: C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_dataset
Klasy: ['HSIL', 'LSIL', 'NSIL']

⚠️  UWAGA: Format prawdopodobieństw w plikach: [p_HSIL, p_LSIL, p_NSIL]

📂 Wczytuję dane treningowe...

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_dataset\train\HSIL\10a.bmp: 480x768 84 cells, 1 HSIL_group, 90.0ms
Speed: 3.7ms preprocess, 90.0ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 768)
Pominięto komórkę 12 - wszystkie features równe 0
Pominięto komórkę 23 - wszystkie features równe 0
Pominięto komórkę 24 - wszystkie features równe 0
Pominięto komórkę 40 - wszystkie features równe 0
Pominięto komórkę 49 - wszystkie features równe 0
Pominięto komórkę 56 - wszystkie features równe 0
Pominięto komórkę 73 - wszystkie features równe 0
Pominięto komórkę 80 - wszystkie features równe 0

image 1/1 C:\Users\aleks\OneDrive\Documents\inzynierka\data\LBC_dataset\train\HSIL\11a.bmp: 480x768 1 ce

Grid Search: 100%|██████████| 50/50 [00:00<00:00, 64.88it/s]


WYNIKI

🏆 NAJLEPSZA KONFIGURACJA:
  Pooling: MEAN
  Wagi [HSIL, LSIL, NILM]: [np.float64(3.0), np.float64(2.0), np.float64(1.0)]

Metryki (średnia ± std z 5 foldów):
  Accuracy:     0.8044 ± 0.0442
  F1 Macro:     0.7425 ± 0.0538
  HSIL Recall:  0.6668 ± 0.0803

📊 Top 10 konfiguracji:
   pooling          weights  mean_accuracy   mean_f1  mean_hsil_recall
15    mean  [3.0, 2.0, 1.0]       0.804429  0.742456          0.666842
5     mean  [2.0, 2.0, 1.0]       0.804382  0.738300          0.595789
17    mean  [3.0, 2.5, 1.0]       0.795338  0.737988          0.646842
3     mean  [2.0, 1.5, 1.0]       0.804382  0.736750          0.626316
19    mean  [3.0, 3.0, 1.0]       0.783077  0.727384          0.616316
27    mean  [4.0, 2.5, 1.0]       0.776970  0.724486          0.666842
29    mean  [4.0, 3.0, 1.0]       0.770909  0.722414          0.656842
39    mean  [5.0, 3.0, 1.0]       0.767786  0.721598          0.686842
7     mean  [2.0, 2.5, 1.0]       0.782984  0.716353          0.535263
13 

In [14]:
results_df

,pooling,w_hsil,w_lsil,w_nilm,weights,mean_accuracy,std_accuracy,mean_f1,std_f1,mean_hsil_recall,std_hsil_recall
15,mean,3.0,2.0,1.0,"[3.0, 2.0, 1.0]",0.804429,0.044242,0.742456,0.053810,0.666842,0.080314
5,mean,2.0,2.0,1.0,"[2.0, 2.0, 1.0]",0.804382,0.037581,0.738300,0.050326,0.595789,0.071210
17,mean,3.0,2.5,1.0,"[3.0, 2.5, 1.0]",0.795338,0.050909,0.737988,0.061598,0.646842,0.098611
3,mean,2.0,1.5,1.0,"[2.0, 1.5, 1.0]",0.804382,0.033548,0.736750,0.048406,0.626316,0.080666
19,mean,3.0,3.0,1.0,"[3.0, 3.0, 1.0]",0.783077,0.048922,0.727384,0.056527,0.616316,0.086216
27,mean,4.0,2.5,1.0,"[4.0, 2.5, 1.0]",0.776970,0.041707,0.724486,0.049305,0.666842,0.080314
29,mean,4.0,3.0,1.0,"[4.0, 3.0, 1.0]",0.770909,0.050912,0.722414,0.061135,0.656842,0.084778
39,mean,5.0,3.0,1.0,"[5.0, 3.0, 1.0]",0.767786,0.043747,0.721598,0.052451,0.686842,0.096834
7,mean,2.0,2.5,1.0,"[2.0, 2.5, 1.0]",0.782984,0.028889,0.716353,0.034436,0.535263,0.058267
13,mean,3.0,1.5,1.0,"[3.0, 1.5, 1.0]",0.792121,0.039122,0.712406,0.055462,0.697368,0.098816


In [ ]:
results_df.to_excel('cv_grid_search_max_vs_mean_HSIL_LSIL_GROUP.xlsx', index=False)

## Attention based slide classification

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os


class SlideDataset(Dataset):
    def __init__(self, data_dict):
        self.slides = data_dict['slides']
        self.labels = data_dict['labels']
        self.filenames = data_dict['filenames']
    
    def __len__(self):
        return len(self.slides)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.slides[idx]), torch.LongTensor([self.labels[idx]])[0], self.filenames[idx]

class AttentionMIL(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, num_classes=3, dropout=0.3):
        super(AttentionMIL, self).__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        # x shape: (batch_size, num_cells, input_dim)
        batch_size = x.shape[0]
        num_cells = x.shape[1]
        
        # Extract features z każdej komórki
        x_flat = x.view(-1, x.shape[2])
        features = self.feature_extractor(x_flat)
        features = features.view(batch_size, num_cells, -1)
        
        # Calculate attention weights
        attention_scores = self.attention(features)
        attention_weights = torch.softmax(attention_scores, dim=1)
        
        # Aggregate features using attention weights
        slide_features = torch.sum(attention_weights * features, dim=1)
        
        # Classify
        output = self.classifier(slide_features)
        
        return output, attention_weights
    

In [19]:
def collate_fn(batch):
    slides, labels, filenames = zip(*batch)
    
    # Znajdź maksymalną liczbę komórek w batch
    max_cells = max([s.shape[0] for s in slides])
    
    # Pad slajdy do tej samej długości
    padded_slides = []
    for slide in slides:
        num_cells = slide.shape[0]
        if num_cells < max_cells:
            padding = torch.zeros(max_cells - num_cells, slide.shape[1])
            padded_slide = torch.cat([slide, padding], dim=0)
        else:
            padded_slide = slide
        padded_slides.append(padded_slide)
    
    return torch.stack(padded_slides), torch.stack(labels), filenames

# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for slides, labels, _ in dataloader:
        slides, labels = slides.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs, _ = model(slides)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(dataloader), 100 * correct / total

# Validation function
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for slides, labels, _ in dataloader:
            slides, labels = slides.to(device), labels.to(device)
            
            outputs, _ = model(slides)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return total_loss / len(dataloader), 100 * correct / total

# Funkcja do ewaluacji z szczegółowymi metrykami
def evaluate_detailed(model, dataloader, device, class_names):
    model.eval()
    all_preds = []
    all_labels = []
    all_filenames = []
    
    with torch.no_grad():
        for slides, labels, filenames in dataloader:
            slides, labels = slides.to(device), labels.to(device)
            outputs, _ = model(slides)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_filenames.extend(filenames)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Confusion matrix
    from sklearn.metrics import confusion_matrix, classification_report
    
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names)
    
    print("\n📊 Confusion Matrix:")
    print(cm)
    print("\n📈 Classification Report:")
    print(report)
    
    # Błędnie sklasyfikowane próbki
    wrong_indices = np.where(all_preds != all_labels)[0]
    if len(wrong_indices) > 0:
        print(f"\n❌ Błędnie sklasyfikowane ({len(wrong_indices)} próbek):")
        for idx in wrong_indices[:10]:  # pokaż pierwsze 10
            print(f"  {all_filenames[idx]}: True={class_names[all_labels[idx]]}, Pred={class_names[all_preds[idx]]}")
    
    return all_preds, all_labels, all_filenames

# Funkcja do wizualizacji attention weights
def visualize_attention(model, slide, filename, device, class_names):
    model.eval()
    with torch.no_grad():
        slide_tensor = torch.FloatTensor(slide).unsqueeze(0).to(device)
        output, attention_weights = model(slide_tensor)
        
        _, predicted = torch.max(output, 1)
        attn = attention_weights.squeeze().cpu().numpy()
        
        print(f"\n🔍 Analiza slajdu: {filename}")
        print(f"  Predicted class: {class_names[predicted.item()]}")
        print(f"  Top 5 important cells (attention weights):")
        top_indices = np.argsort(attn.flatten())[-5:][::-1]
        for i, idx in enumerate(top_indices):
            if idx < len(slide):
                print(f"    Cell {idx}: weight={attn[idx]:.4f}, probs={slide[idx]}")
        
        return predicted.item(), attn


In [43]:
import numpy as np  # <--- DODANE (jeśli nie masz)
from torch.utils.data import DataLoader, WeightedRandomSampler  # <--- DODANE (import samplera)

LEARNING_RATE = 0.001
NUM_EPOCHS = 50
HIDDEN_DIM = 64
DROPOUT = 0.3
BATCH_SIZE = 4

from sklearn.model_selection import train_test_split
    
print("\n🔀 Tworzę split train/validation (80/20)...")
train_indices, val_indices = train_test_split(
        list(range(len(train_data['slides']))),
        test_size=0.2,
        random_state=42,
        stratify=train_data['labels']
    )
    
train_split_data = {
    'slides': [train_data['slides'][i] for i in train_indices],
    'labels': train_data['labels'][train_indices],
    'filenames': [train_data['filenames'][i] for i in train_indices]
    }
    
val_data = {
    'slides': [train_data['slides'][i] for i in val_indices],
    'labels': train_data['labels'][val_indices],
    'filenames': [train_data['filenames'][i] for i in val_indices]
    }
    
print(f"  Train: {len(train_split_data['slides'])} slajdów")
print(f"  Val:   {len(val_data['slides'])} slajdów")
print(f"  Test:  {len(test_data['slides'])} slajdów")
    
train_dataset = SlideDataset(train_split_data)
val_dataset = SlideDataset(val_data)
test_dataset = SlideDataset(test_data)

# --- Początek implementacji WeightedRandomSampler --- # <--- DODANE

print("\n⚖️ Tworzę WeightedRandomSampler dla zbioru treningowego...")

# 1. Pobierz etykiety ze zbioru treningowego
train_labels = train_dataset.labels 

# 2. Oblicz liczebność każdej klasy
class_counts = np.bincount(train_labels)
num_samples = len(train_labels)
num_classes = len(class_counts)

# 3. Oblicz wagi dla każdej KLASY (formuła "balanced")
class_weights = num_samples / (num_classes * class_counts)
print(f"  Obliczone wagi klas: {class_weights}")

# 4. Utwórz listę wag dla każdej PRÓBKI
sample_weights = [class_weights[label] for label in train_labels]

# 5. Stwórz sampler
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(sample_weights),
    replacement=True  # replacement=True pozwala na nadpróbkowanie (oversampling)
)

# --- Koniec implementacji WeightedRandomSampler --- # <--- DODANE

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    # shuffle=True,  <--- ZMIENIONE (nie można używać shuffle z samplerem)
    shuffle=False,   # <--- ZMIENIONE (sampler sam zajmuje się losowością)
    sampler=sampler, # <--- DODANE (przekazujemy nasz sampler)
    collate_fn=collate_fn
    )

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
    )

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
    )

print("✅ Dataloadery gotowe (train_loader używa samplera).")


🔀 Tworzę split train/validation (80/20)...
  Train: 261 slajdów
  Val:   66 slajdów
  Test:  82 slajdów

⚖️ Tworzę WeightedRandomSampler dla zbioru treningowego...
  Obliczone wagi klas: [     1.1013       2.175     0.61268]
✅ Dataloadery gotowe (train_loader używa samplera).


In [44]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️ Używam urządzenia: {device}")
    
model = AttentionMIL(
        input_dim=3, 
        hidden_dim=HIDDEN_DIM, 
        num_classes=len(CLASS_NAMES), 
        dropout=DROPOUT).to(device)
    
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
# Training loop
best_val_acc = 0
patience = 50
patience_counter = 0
    
print("\n🚀 Rozpoczynam trening...")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
        
    # Early stopping based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_attention_mil_model.pth')
        patience_counter = 0
        print(f"✓ Zapisano najlepszy model (val acc: {best_val_acc:.2f}%)")
    else:
        patience_counter += 1
        
    if (epoch + 1) % 10 == 0:
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
        
    # Early stopping
    if patience_counter >= patience:
        print(f"\n⏹️ Early stopping - brak poprawy przez {patience} epok. Koniec treningu na epoce {epoch+1}.")
        break


# Load best model i szczegółowa ewaluacja
print("\n" + "="*60)
print("📊 FINALNA EWALUACJA")
print("="*60)
model.load_state_dict(torch.load('best_attention_mil_model.pth'))
evaluate_detailed(model, test_loader, device, CLASS_NAMES)
    
# Przykład wizualizacji attention dla pierwszego slajdu testowego
if len(test_data['slides']) > 0:
    print("\n" + "="*60)
    print("👁️ PRZYKŁADOWA WIZUALIZACJA ATTENTION")
    print("="*60)
    visualize_attention(
        model, 
        test_data['slides'][0], 
        test_data['filenames'][0],
        device, 
        CLASS_NAMES
    )



🖥️ Używam urządzenia: cpu

🚀 Rozpoczynam trening...
✓ Zapisano najlepszy model (val acc: 59.09%)
✓ Zapisano najlepszy model (val acc: 66.67%)
✓ Zapisano najlepszy model (val acc: 72.73%)
✓ Zapisano najlepszy model (val acc: 74.24%)
✓ Zapisano najlepszy model (val acc: 78.79%)

Epoch 10/50
  Train - Loss: 0.6379, Acc: 76.63%
  Val   - Loss: 0.5691, Acc: 72.73%
✓ Zapisano najlepszy model (val acc: 81.82%)
✓ Zapisano najlepszy model (val acc: 83.33%)

Epoch 20/50
  Train - Loss: 0.5700, Acc: 75.86%
  Val   - Loss: 0.4649, Acc: 77.27%
✓ Zapisano najlepszy model (val acc: 84.85%)
✓ Zapisano najlepszy model (val acc: 87.88%)

Epoch 30/50
  Train - Loss: 0.5468, Acc: 78.54%
  Val   - Loss: 0.3965, Acc: 81.82%

Epoch 40/50
  Train - Loss: 0.5859, Acc: 77.39%
  Val   - Loss: 0.3759, Acc: 81.82%

Epoch 50/50
  Train - Loss: 0.6596, Acc: 71.65%
  Val   - Loss: 0.3463, Acc: 81.82%

📊 FINALNA EWALUACJA

📊 Confusion Matrix:
[[16  7  1]
 [ 3  9  1]
 [ 0  3 42]]

📈 Classification Report:
            

In [27]:
test_data['slides'][2]

[array([    0.99041,   0.0084711,   0.0011145]),
 array([    0.78616,     0.21268,   0.0011586]),
 array([    0.87998,     0.11351,   0.0065141]),
 array([    0.92606,     0.07203,   0.0019153]),
 array([     0.9952,   0.0036367,   0.0011634]),
 array([  0.0049988,      0.9863,    0.008705])]

In [28]:
slide = [np.array([ 2.7886e-05,     0.99584,   0.0041329]), np.array([  0.0024515,     0.98735,    0.010195]), np.array([  0.0021589,      0.9906,   0.0072459]), np.array([  8.206e-06,      0.9979,   0.0020924]), np.array([  0.0020331,       0.992,   0.0059672]), np.array([    0.89986,    0.060451,    0.039688]), np.array([   0.015794,     0.98037,   0.0038368]), np.array([ 3.6391e-05,     0.99538,   0.0045857])]
visualize_attention(model, slide, 'test', device, CLASS_NAMES)


🔍 Analiza slajdu: test
  Predicted class: LSIL
  Top 5 important cells (attention weights):
    Cell 3: weight=0.1294, probs=[  8.206e-06      0.9979   0.0020924]
    Cell 0: weight=0.1292, probs=[ 2.7886e-05     0.99584   0.0041329]
    Cell 7: weight=0.1292, probs=[ 3.6391e-05     0.99538   0.0045857]
    Cell 4: weight=0.1288, probs=[  0.0020331       0.992   0.0059672]
    Cell 2: weight=0.1287, probs=[  0.0021589      0.9906   0.0072459]


(1,
 array([     0.1292,     0.12838,     0.12871,     0.12941,     0.12885,    0.098377,     0.12793,     0.12915], dtype=float32))

In [29]:
slide = [np.array([   0.017954,     0.97584,   0.0062091]), np.array([    0.11891,     0.55086,     0.33023]), np.array([    0.99023,   0.0073096,   0.0024576]), np.array([    0.78636,     0.21241,   0.0012248]), np.array([    0.18094,     0.33201,     0.48705]), np.array([    0.99505,   0.0028757,   0.0020769]), np.array([    0.22094,     0.29442,     0.48464]), np.array([    0.94493,     0.05062,   0.0044457]), np.array([    0.99431,   0.0042377,   0.0014509]), np.array([     0.9954,   0.0017875,    0.002812])]
visualize_attention(model, slide, 'test', device, CLASS_NAMES)


🔍 Analiza slajdu: test
  Predicted class: HSIL
  Top 5 important cells (attention weights):
    Cell 0: weight=0.1414, probs=[   0.017954     0.97584   0.0062091]
    Cell 8: weight=0.1177, probs=[    0.99431   0.0042377   0.0014509]
    Cell 5: weight=0.1177, probs=[    0.99505   0.0028757   0.0020769]
    Cell 9: weight=0.1176, probs=[     0.9954   0.0017875    0.002812]
    Cell 2: weight=0.1174, probs=[    0.99023   0.0073096   0.0024576]


(0,
 array([    0.14145,    0.073046,     0.11744,     0.11129,    0.045644,     0.11765,    0.042719,     0.11548,      0.1177,     0.11758], dtype=float32))

In [31]:
slide = [np.array([   0.071536,    0.072288,     0.85618]), np.array([  0.0019841,    0.002361,     0.99565]), np.array([  0.0060997,    0.010887,     0.98301]), np.array([  0.0021393,   0.0049249,     0.99294]), np.array([   0.015889,    0.035407,      0.9487]), np.array([  0.0022652,   0.0023194,     0.99542]), np.array([   0.015033,    0.013413,     0.97155]), np.array([    0.03418,     0.14673,     0.81909])]
visualize_attention(model, slide, 'test', device, CLASS_NAMES)


🔍 Analiza slajdu: test
  Predicted class: NSIL
  Top 5 important cells (attention weights):
    Cell 1: weight=0.1297, probs=[  0.0019841    0.002361     0.99565]
    Cell 5: weight=0.1297, probs=[  0.0022652   0.0023194     0.99542]
    Cell 3: weight=0.1295, probs=[  0.0021393   0.0049249     0.99294]
    Cell 2: weight=0.1286, probs=[  0.0060997    0.010887     0.98301]
    Cell 6: weight=0.1273, probs=[   0.015033    0.013413     0.97155]


(2,
 array([    0.11589,      0.1297,     0.12856,     0.12949,     0.12546,     0.12968,     0.12732,     0.11391], dtype=float32))

In [33]:
slide = [np.array([  0.0095552,     0.78267,     0.20777]), np.array([    0.35931,     0.63454,   0.0061475]), np.array([    0.99708,   0.0013094,     0.00161]), np.array([   0.039165,    0.059247,     0.90159]), np.array([  0.0047941,   0.0086451,     0.98656]), np.array([  0.0025545,   0.0095348,     0.98791]), np.array([ 1.5846e-05,     0.99705,   0.0029352]), np.array([  0.0036693,   0.0096488,     0.98668]), np.array([    0.99525,   0.0036332,   0.0011165]), np.array([    0.56378,     0.41279,    0.023428]), np.array([  0.0023887,   0.0023697,     0.99524]), np.array([      0.996,   0.0022685,    0.001731]), np.array([    0.99725,   0.0015455,   0.0012033]), np.array([    0.79784,     0.10787,    0.094291]), np.array([   0.019801,   0.0076358,     0.97256]), np.array([   0.013904,    0.012752,     0.97334])]
visualize_attention(model, slide, 'test', device, CLASS_NAMES)


🔍 Analiza slajdu: test
  Predicted class: HSIL
  Top 5 important cells (attention weights):
    Cell 6: weight=0.0928, probs=[ 1.5846e-05     0.99705   0.0029352]
    Cell 0: weight=0.0784, probs=[  0.0095552     0.78267     0.20777]
    Cell 12: weight=0.0754, probs=[    0.99725   0.0015455   0.0012033]
    Cell 2: weight=0.0754, probs=[    0.99708   0.0013094     0.00161]
    Cell 11: weight=0.0753, probs=[      0.996   0.0022685    0.001731]


(0,
 array([   0.078354,    0.072052,    0.075391,    0.044412,    0.047374,    0.047446,    0.092848,    0.047394,    0.075331,    0.064281,    0.047658,    0.075344,     0.07541,    0.063066,    0.046775,    0.046863], dtype=float32))

## KFOLD

In [46]:
def get_detailed_metrics(model, dataloader, device, class_names):
    """
    Zwraca szczegółowe metryki (jako słownik) dla danego modelu i dataloadera.
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for slides, labels, _ in dataloader:
            slides, labels = slides.to(device), labels.to(device)
            outputs, _ = model(slides)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Tłumienie ostrzeżeń sklearn (na wypadek zerowej precyzji/recall)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        report = classification_report(
            all_labels, 
            all_preds, 
            target_names=class_names, 
            output_dict=True  # Zwraca słownik
        )
    
    return report

In [48]:
import copy


print("\n🔄 Rozpoczynam Walidację Krzyżową (Stratified K-Fold)...")

# 4. Przygotowanie danych do K-Fold
all_slides = train_data['slides']
all_labels = np.array(train_data['labels']) # Ważne: Konwersja na np.array
all_filenames = train_data['filenames']
X_indices = np.arange(len(all_slides))
y_labels = all_labels

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 5. Listy do przechowywania wyników
fold_val_accuracies = []
fold_hsil_recalls = [] # Do śledzenia recall HSIL

# --- Główna Pętla K-Fold ---
for fold, (train_indices, val_indices) in enumerate(skf.split(X_indices, y_labels)):
    
    print(f"\n--- FOLD {fold + 1}/{N_SPLITS} ---")
    
    # 6. Stwórz dane dla tego foldu
    fold_train_data = {
        'slides': [all_slides[i] for i in train_indices],
        'labels': all_labels[train_indices],
        'filenames': [all_filenames[i] for i in train_indices]
    }
    fold_val_data = {
        'slides': [all_slides[i] for i in val_indices],
        'labels': all_labels[val_indices],
        'filenames': [all_filenames[i] for i in val_indices]
    }

    # 7. Stwórz Datasety
    train_dataset = SlideDataset(fold_train_data)
    val_dataset = SlideDataset(fold_val_data)
    
    print(f"  Train: {len(train_dataset)} slajdów, Val: {len(val_dataset)} slajdów")

    # 8. Stwórz WeightedRandomSampler DLA TEGO FOLDU
    train_labels_fold = train_dataset.labels
    class_counts = np.bincount(train_labels_fold)
    num_samples = len(train_labels_fold)
    
    # Oblicz wagi, unikając dzielenia przez zero (jeśli jakaś klasa ma 0 próbek)
    class_weights = num_samples / (len(class_counts) * np.where(class_counts == 0, 1, class_counts))
    # Ustaw wagę na 0 dla klas, które nie istnieją
    class_weights[class_counts == 0] = 0 
    
    sample_weights = [class_weights[label] for label in train_labels_fold]
    
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float),
        num_samples=len(sample_weights),
        replacement=True
    )

    # 9. Stwórz DataLoadery DLA TEGO FOLDU
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        shuffle=False, # Sampler robi shuffle
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    # 10. Reinicjalizuj model i optymalizator
    print("  Inicjalizuję nowy model...")
    model = AttentionMIL(input_dim=3, hidden_dim=HIDDEN_DIM, num_classes=len(CLASS_NAMES), dropout=DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss() # Standardowa strata (bo mamy sampler)

    best_val_acc = 0.0
    best_model_state = None # Do przechowania wag najlepszego modelu

    # 11. Pętla treningowa dla epok
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        if (epoch + 1) % 10 == 0: # Drukuj co 10 epok
            print(f'  Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict()) # Zapisz wagi w pamięci

    # --- Koniec Pętli Epok dla Foldu ---
    print(f"  --- Najlepsza dokładność walidacyjna dla foldu {fold+1}: {best_val_acc:.2f}% ---")

    # 12. Ewaluacja najlepszego modelu z tego foldu
    if best_model_state:
        model.load_state_dict(best_model_state) # Wczytaj najlepsze wagi
        val_report = get_detailed_metrics(model, val_loader, device, CLASS_NAMES)
        
        # Wyciągnij recall dla klasy 'HSIL'
        hsil_recall = val_report['HSIL']['recall']
        
        print(f"  📈 Recall HSIL dla najlepszego modelu: {hsil_recall*100:.2f}%")
        fold_hsil_recalls.append(hsil_recall)
    else:
        print("  --- Nie udało się zapisać najlepszego modelu (być może Acc=0.0) ---")
        fold_hsil_recalls.append(0.0)

    fold_val_accuracies.append(best_val_acc)

# --- Koniec Pętli K-Fold ---

# 13. Oblicz i wyświetl średnie wyniki
mean_acc = np.mean(fold_val_accuracies)
std_acc = np.std(fold_val_accuracies)
mean_hsil_recall = np.mean(fold_hsil_recalls)
std_hsil_recall = np.std(fold_hsil_recalls)

print(f"\n\n--- Podsumowanie Walidacji Krzyżowej ({N_SPLITS}-krotnej) ---")
print(f"Wyniki dokładności (Accuracy) dla każdego foldu: {[f'{acc:.2f}%' for acc in fold_val_accuracies]}")
print(f"📈 Średnia dokładność walidacyjna: {mean_acc:.2f}% ± {std_acc:.2f}%")
print("\n")
print(f"Wyniki Recall (HSIL) dla każdego foldu: {[f'{rec*100:.2f}%' for rec in fold_hsil_recalls]}")
print(f"🎯 Średni Recall dla HSIL: {mean_hsil_recall*100:.2f}% ± {std_hsil_recall*100:.2f}%")
print("--------------------------------------------------")


🔄 Rozpoczynam Walidację Krzyżową (Stratified K-Fold)...

--- FOLD 1/5 ---
  Train: 261 slajdów, Val: 66 slajdów
  Inicjalizuję nowy model...
  Epoch 10/50 | Train Loss: 0.6712 Acc: 70.88% | Val Loss: 0.6026 Acc: 75.76%
  Epoch 20/50 | Train Loss: 0.5826 Acc: 73.95% | Val Loss: 0.6329 Acc: 75.76%
  Epoch 30/50 | Train Loss: 0.4780 Acc: 76.63% | Val Loss: 0.6225 Acc: 77.27%
  Epoch 40/50 | Train Loss: 0.4952 Acc: 74.33% | Val Loss: 0.6641 Acc: 74.24%
  Epoch 50/50 | Train Loss: 0.5097 Acc: 78.16% | Val Loss: 0.6776 Acc: 75.76%
  --- Najlepsza dokładność walidacyjna dla foldu 1: 78.79% ---
  📈 Recall HSIL dla najlepszego modelu: 70.00%

--- FOLD 2/5 ---
  Train: 261 slajdów, Val: 66 slajdów
  Inicjalizuję nowy model...
  Epoch 10/50 | Train Loss: 0.5128 Acc: 78.93% | Val Loss: 0.4652 Acc: 77.27%
  Epoch 20/50 | Train Loss: 0.5685 Acc: 76.25% | Val Loss: 0.4576 Acc: 75.76%
  Epoch 30/50 | Train Loss: 0.5423 Acc: 75.86% | Val Loss: 0.4443 Acc: 78.79%
  Epoch 40/50 | Train Loss: 0.5879 Acc: